In [1]:
import sksurv
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
import pandas as pd
import numpy as np
import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
import os
import kagglehub
from dataclasses import asdict

from Utils.Config import Config, TrialResult
from Utils.utils import (
    set_seed,
    make_surv_y,
    make_oof_predictions,
    create_features,
    load_config_yaml,
    HORIZONS
)

In [2]:
storage = JournalStorage(
    JournalFileBackend("gbsa_journal.log")
)

study = optuna.load_study(
    study_name="gbsa_survival",
    storage=storage,
)

In [3]:
best_trial = study.best_trial
best_trial_num = best_trial.number
print("Best trial:")
print(f"  Number: {best_trial.number}")
print(f"  Value: {best_trial.value}")
print("  Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

Best trial:
  Number: 106
  Value: 0.9629674137093398
  Params: 
    n_estimators: 417
    learning_rate: 0.11063731185373184
    subsample: 0.5905037758154414
    max_depth: 6
    max_features: sqrt
    max_leaf_nodes: 31
    min_samples_split: 4
    min_samples_leaf: 8
    min_impurity_decrease: 0.02904783931441647
    ccp_alpha: 0.004992133979210339
    dropout_rate: 0.0007471276215608402
    validation_fraction: 0.1897124693474042
    n_iter_no_change: None
    tol: 0.0004912572712305443


In [4]:
trial_dir = os.path.join("Trials", "GBSA")
best_trial_path = os.path.join(trial_dir, f'trial_{best_trial.number}', 'config.yaml')
print(f"Best trial config path: {best_trial_path}")
print(f"Best trial config exists: {os.path.exists(best_trial_path)}")

Best trial config path: Trials/GBSA/trial_106/config.yaml
Best trial config exists: True


In [5]:
COMP_DIR = kagglehub.competition_download('WiDSWorldWide_GlobalDathon26')
metadata_path = os.path.join(COMP_DIR, 'metaData.csv')
train_path = os.path.join(COMP_DIR, 'train.csv')
test_path = os.path.join(COMP_DIR, 'test.csv')

In [6]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
train_processed = create_features(train_df)
test_processed = create_features(test_df)

y_full = make_surv_y(
    event=train_processed['event'],
    time=train_processed['time_to_hit_hours']
)

event_horizon = np.array(HORIZONS).copy()
event_horizon[-1] = min(event_horizon[-1], train_processed['time_to_hit_hours'].max() - 1e-6)

print("Event horizons:", event_horizon)

Event horizons: [12.         24.         48.         66.99447313]


In [7]:
config = load_config_yaml(best_trial_path)
print("Loaded config:")
print(config)

Loaded config:
Config(seed=42, cv_n_splits=5, cv_n_repeats=10, gbsa_config=GBSAConfig(loss='coxph', n_estimators=417, learning_rate=0.11063731185373184, subsample=0.5905037758154414, max_depth=6, max_features='sqrt', max_leaf_nodes=31, min_samples_split=4, min_samples_leaf=8, min_weight_fraction_leaf=0.0, min_impurity_decrease=0.02904783931441647, criterion='friedman_mse', ccp_alpha=0.004992133979210339, dropout_rate=0.0007471276215608402, validation_fraction=0.1897124693474042, n_iter_no_change=None, tol=0.0004912572712305443, random_state=42, warm_start=False, verbose=0), preprocessing_config=PreprocessingConfig(eps=1e-06, min_speed=0.01, max_hours=9999.0))


In [8]:
seed = config.seed
set_seed(seed)
model_params = config.gbsa_config

n_splits = config.cv_n_splits
n_repeats = config.cv_n_repeats

In [ ]:
model = GradientBoostingSurvivalAnalysis(**asdict(model_params))

oof_pred = make_oof_predictions(
    model=model,
    data=train_processed,
    horizons=event_horizon,
    seed=seed,
    n_splits=n_splits,
    n_repeats=n_repeats
)
print(oof_pred)

KFold OOF:   0%|          | 0/50 [00:00<?, ?it/s]